# Welche Gene schaltet eine Maus an, wenn sie Milch macht?

Die **Milchdrüse** verändert sich enorm in der Schwangerschaft und wenn die Mutter ihre Jungen säugt (**Laktation**). Sie besteht aus zwei Arten von Zellen:

* **luminale Zellen** kleiden die Milchgänge innen aus und **produzieren** während der Laktation **die Milch**,
* **basale Zellen** sitzen außen herum und **pressen** die Milch heraus, wie ein Muskel.

Forschende (Fu et al. 2015) haben beide Zelltypen aus Mäusen in drei Zuständen entnommen: **virgin** (jungfräulich, nie trächtig), **pregnant** (trächtig) und **lactating** (säugend), mit zwei Mäusen pro Kombination. Die RNA dieser **12 Proben** wurde sequenziert.

### Unsere Forschungsfrage

> Welche Gene schalten die **luminalen Zellen** an (oder aus), wenn sie von der **Trächtigkeit** zur **Milchproduktion** übergehen? Können wir aus diesen Genen erklären, woraus Milch besteht?

### Der Plan

```mermaid
flowchart LR
    A[1 Daten<br>ansehen] --> B[2 Gruppen<br>festlegen] --> C[3 Vorbereiten:<br>filtern, normalisieren] --> D[4 Design und<br>Kontraste] --> E[5 Testen<br>mit limma] --> F[6 Deuten]
```

Führe die Zellen der Reihe nach von oben aus, mit **Umschalt + Enter**.

In [ ]:
setLanguage("de")   # Sprache der Ausgaben und Grafiken

## 1 Die Daten ansehen

`loadDataset` lädt die Daten. Sie bestehen aus drei Tabellen: `samples` (was jede Probe ist), `counts` (die Reads) und `genes` (Gennamen). Sehen wir uns zuerst die Proben an.

In [ ]:
data <- loadDataset("milk")
data$samples

Jede Probe hat einen Namen (`MCL1-DG` …), einen Zelltyp (`celltype`) und einen Zustand (`state`). Die Einträge in der Tabelle sind englisch: `lactating` heißt „säugend“, `pregnant` „trächtig“, `virgin` „jungfräulich“.

Jetzt die Counts. `head()` zeigt nur die ersten 6 Zeilen; die ganze Tabelle ist viel länger.

In [ ]:
head(data$counts)

Jede **Zeile** ist ein Gen (die Zahlen links sind Gen-IDs aus der NCBI-Datenbank), jede **Spalte** eine Probe, und jede Zahl sagt, wie viele Reads dieses Gens in dieser Probe gefunden wurden. Die Tabelle `genes` übersetzt die IDs in Gennamen:

In [ ]:
head(data$genes)

**Aufgabe:** Wie viele Gene und wie viele Proben enthält die Zähltabelle? `dim()` gibt die Größe einer Tabelle an: zuerst die Zahl der Zeilen, dann der Spalten.

In [ ]:
dim(___)

✏️ **Deine Antwort:** *(Doppelklick hier, um zu schreiben)*



Für die Analyse packen wir die drei Tabellen in einen Behälter, eine `DGEList` aus dem Paket **edgeR**. Sie zählt auch die Reads jeder Probe zusammen: die **Bibliotheksgröße** (library size).

In [ ]:
dge <- DGEList(counts = data$counts, samples = data$samples, genes = data$genes)
dge$samples

In [ ]:
plotLibrarySizes(dge, colour_by = "celltype")

**Frage:** Haben alle Proben gleich viele Reads? Warum wäre es falsch, die rohen Counts eines Gens zwischen zwei Proben zu vergleichen?

✏️ **Deine Antwort:** *(Doppelklick hier, um zu schreiben)*



**Aufgabe:** Viele Gene werden in diesen Zellen gar nicht benutzt. Wie viele Gene haben **0 Reads in allen 12 Proben**? `rowSums()` addiert jede Zeile, und `sum(... == x)` zählt, wie oft etwas gleich `x` ist.

In [ ]:
sum(rowSums(dge$counts) == ___)

✏️ **Deine Antwort:** *(Doppelklick hier, um zu schreiben)*



## 2 Die Gruppen festlegen

Zum Vergleichen muss limma wissen, welche Proben zusammengehören. Jede Kombination aus Zelltyp und Zustand ist eine **Gruppe**, zum Beispiel `luminal_lactating`. Wir kleben die beiden Spalten mit `paste` zusammen und speichern das Ergebnis in der Spalte `group`.

In [ ]:
group <- paste(dge$samples$celltype, dge$samples$state, sep = "_")
dge$samples$group <- factor(group, levels = unique(group))
table(dge$samples$group)

**Frage:** Wie viele Proben hat jede Gruppe? Warum benutzen Forschende mehrere Mäuse pro Gruppe (**Replikate**) statt nur einer?

✏️ **Deine Antwort:** *(Doppelklick hier, um zu schreiben)*



Sehen wir uns ein einzelnes Gen an. **Csn2** ist das Gen für **β-Casein**, das wichtigste Protein in der Milch. `plotGene` zeigt seine Expression in jeder Probe als **log2 Counts pro Million**: 1 höher bedeutet doppelt so viel. Wo erwartest du es hoch?

In [ ]:
plotGene(dge, "Csn2", group_by = "group")

**Aufgabe:** Prüfe mit deinem Biologiewissen, ob die Proben richtig beschriftet sind. **Krt5** (Keratin 5) ist ein typisches Protein der basalen Zellen, **Krt8** (Keratin 8) der luminalen Zellen. Zeichne eins davon.

In [ ]:
plotGene(dge, "___", group_by = "group")

✏️ **Deine Antwort:** *(Doppelklick hier, um zu schreiben)*



## 3 Die Daten vorbereiten

### Filtern

Gene mit nur einer Handvoll Reads lassen sich nicht zuverlässig testen: 0 gegen 2 Reads könnte gut Zufall sein. `filterByExpr` behält nur Gene, die genug Reads in mindestens so vielen Proben haben, wie die kleinste Gruppe hat (hier 2). `TRUE` heißt behalten.

In [ ]:
keep <- filterByExpr(dge)
table(keep)

In [ ]:
dge <- dge[keep, , keep.lib.sizes = FALSE]
dim(dge)

### Normalisieren

Jetzt sehen wir uns die Verteilung **aller** Gene in jeder Probe an. Jeder Kasten zeigt die mittlere Hälfte der Gene; der Strich im Kasten ist der Median. Wenn die Proben vergleichbar sind, sollten die Kästen etwa gleich hoch liegen.

In [ ]:
plotDistributions(dge, colour_by = "group", title = "Vor dem Normalisieren")

Sieh dir die beiden säugenden luminalen Proben an (`MCL1-LE`, `MCL1-LF`): Ihre Kästen liegen **tiefer**. Sind wirklich alle ihre Gene weniger aktiv? Nein! Diese Zellen stecken einen riesigen Teil ihrer RNA in ein paar Milchgene. Berechne den Anteil der Reads, den allein Csn2 (Gen-ID 12991) in der Probe `MCL1-LE` hat:

In [ ]:
data$counts["12991", "MCL1-LE"] / sum(data$counts[, "MCL1-LE"])   # Anteil an allen Reads

Etwa **10 %** aller Reads stammen von diesem einen Gen. Dann bleiben für alle anderen Gene weniger Reads übrig, und sie *sehen* niedriger aus. Die **TMM-Normalisierung** in `normLibSizes` gleicht das aus: Sie berechnet für jede Probe einen Korrekturfaktor (`norm.factors`) aus den vielen Genen, die sich nicht verändern.

In [ ]:
dge <- normLibSizes(dge)
dge$samples[, c("group", "lib.size", "norm.factors")]

In [ ]:
plotDistributions(dge, colour_by = "group", title = "Nach dem Normalisieren")

**Frage:** Wie viele Gene bleiben nach dem Filtern übrig? Welche Proben bekommen den kleinsten Korrekturfaktor, und was hat sich im Kastendiagramm verändert?

✏️ **Deine Antwort:** *(Doppelklick hier, um zu schreiben)*



### Die Probenkarte (MDS)

Bevor wir einzelne Gene testen, sehen wir uns an, wie ähnlich sich die Proben **insgesamt** sind. Ein **MDS-Plot** setzt jede Probe auf eine Karte: Proben, deren ~16.000 Gene ähnlich exprimiert sind, liegen nah beieinander. **Richtung 1** (links–rechts) ist die Richtung, in der sich die Proben am stärksten unterscheiden; die Prozentzahl sagt, wie viel aller Unterschiede sie zeigt.

In [ ]:
plotSamples(dge, colour_by = "celltype", label_by = "state")

**Aufgabe:** Färbe die Proben jetzt nach Zustand ein und beschrifte sie mit dem Zelltyp.

In [ ]:
plotSamples(dge, colour_by = "___", label_by = "___")

**Frage:** Was trennt die Proben entlang Richtung 1, was entlang Richtung 2? Liegen die beiden Replikate jeder Gruppe nah beieinander? Warum ist das eine gute Nachricht?

✏️ **Deine Antwort:** *(Doppelklick hier, um zu schreiben)*



## 4 Design und Kontraste

### Die Design-Matrix

Die **Design-Matrix** sagt limma, zu welcher Gruppe jede Probe gehört: eine Spalte pro Gruppe, mit einer 1, wenn die Probe dazugehört.

In [ ]:
design <- model.matrix(~0 + group, data = dge$samples)
colnames(design) <- levels(dge$samples$group)
design

### Kontraste: Welche Gruppen vergleichen wir?

Ein **Kontrast** ist der Vergleich, der dich interessiert, geschrieben als Subtraktion: `A - B` fragt „wie unterscheidet sich A von B?“. Ein **positiver** log Fold Change bedeutet dann **höher in A**.

Unsere Frage vergleicht **säugende** mit **trächtigen** luminalen Zellen. Warum nicht einfach alle säugenden mit allen trächtigen Proben? Weil basale Zellen keine Milch machen, und wenn wir sie mischen, verwischt das Ergebnis: Der MDS-Plot hat gezeigt, dass der Zelltyp der größte Unterschied von allen ist.

**Aufgabe:** Fülle die beiden Kontraste aus:
1. `lactating_vs_pregnant`: luminale Zellen, säugend verglichen mit trächtig,
2. `luminal_vs_basal`: bei jungfräulichen Mäusen, luminale verglichen mit basalen Zellen.

Benutze die Gruppennamen genau so wie in der Design-Matrix.

In [ ]:
contrasts <- makeContrasts(
    lactating_vs_pregnant = ___ - ___,
    luminal_vs_basal      = ___ - ___,
    levels = design)
contrasts

**Frage:** Was bedeutet im Kontrast `lactating_vs_pregnant` ein log Fold Change von **+3** für ein Gen? Und **−1**? Welchen Kontrast würdest du schreiben, um die Gene zu finden, die sich in **basalen** Zellen in der Trächtigkeit verändern?

✏️ **Deine Antwort:** *(Doppelklick hier, um zu schreiben)*



## 5 Jedes Gen mit limma testen

Drei Schritte:

1. **`voom`** wandelt die Counts in log2 Counts pro Million um und schätzt, wie zuverlässig jeder Wert ist: Gene mit wenigen Reads schwanken stärker und bekommen deshalb weniger Gewicht. Der Plot zeigt das: Je weniger Reads (links), desto größer die Schwankung (oben).
2. **`lmFit`** berechnet für jedes Gen den Mittelwert in jeder Gruppe, und **`contrasts.fit`** die Unterschiede, nach denen wir in den Kontrasten gefragt haben.
3. **`eBayes`** testet, ob jeder Unterschied größer ist als die natürliche Schwankung zwischen den Replikaten. Mit nur 2 Replikaten lässt sich diese Schwankung für ein einzelnes Gen schlecht schätzen, also „leiht“ sich limma Information von allen anderen Genen. Deshalb funktioniert limma auch bei kleinen Experimenten.

In [ ]:
v <- voom(dge, design, plot = TRUE)

In [ ]:
fit <- lmFit(v, design)
fit <- contrasts.fit(fit, contrasts)
fit <- eBayes(fit)

### Wie viele Gene haben sich verändert?

Für jedes Gen berechnet limma einen **p-Wert**: Wie wahrscheinlich ist ein so großer Unterschied, wenn sich das Gen in Wirklichkeit gar nicht verändert hat? Klein = überzeugend.

Aber Vorsicht: Wir testen etwa 16.000 Gene. Mit der üblichen Grenze p < 0,05 würden etwa 5 % der Gene, die sich **nicht** verändern, trotzdem zufällig signifikant aussehen: Das wären 800 falsche Treffer! Deshalb gibt limma auch einen **adjustierten p-Wert** (`adj.P.Val`) an. Nehmen wir alle Gene mit adjustiertem p < 0,05, dann sind nur etwa 5 % **dieser Liste** falsche Treffer.

`decideTests` zählt die Gene mit adjustiertem p < 0,05: hoch (`Up`), runter (`Down`) oder nicht signifikant (`NotSig`).

In [ ]:
summary(decideTests(fit))

**Frage:** Wie viele Gene gehen von der Trächtigkeit zur Laktation hoch, wie viele runter? Welcher der beiden Kontraste hat mehr veränderte Gene, und passt das zum MDS-Plot?

✏️ **Deine Antwort:** *(Doppelklick hier, um zu schreiben)*



## 6 Die Ergebnisse deuten

### Der Vulkan-Plot

Im **Vulkan-Plot** ist jedes Gen ein Punkt. Links–rechts ist der log Fold Change (links: niedriger beim Säugen, rechts: höher), und nach oben, wie sicher das Ergebnis ist. Die spannendsten Gene sind oben links und oben rechts: eine große Veränderung **und** überzeugend. Farbige Punkte sind signifikant und ändern sich mindestens um das 2-fache (gestrichelte Linien).

In [ ]:
plotVolcano(fit, "lactating_vs_pregnant")

### Die Top-Gene

`showTop` listet die überzeugendsten Gene mit Namen und Beschreibung auf. `change` übersetzt den log Fold Change zurück in „-mal höher/niedriger“.

In [ ]:
showTop(fit, "lactating_vs_pregnant", n = 15, direction = "up")

**Aufgabe:** Wähle ein Gen aus der Liste, das dich interessiert, und sieh es dir in allen Gruppen an.

Finde heraus, was es tut: Suche seinen Namen auf [GeneCards](https://www.genecards.org) (für menschliche Gene; das Maus-Gen tut meist dasselbe), [NCBI Gene](https://www.ncbi.nlm.nih.gov/gene) oder [UniProt](https://www.uniprot.org), oder denke über die Beschreibung nach. *Tipp: Milch enthält Eiweiß (Caseine), Fett, Milchzucker (Laktose), Calcium, Zink und Phosphat.*

In [ ]:
plotGene(dge, "___", group_by = "group")

**Frage:** Kannst du die Top-Gene damit verbinden, woraus Milch besteht? Finde mindestens drei Beispiele.

✏️ **Deine Antwort:** *(Doppelklick hier, um zu schreiben)*



Wo sind die berühmten Milchgene wie **Wap** (Molkenprotein) und **Lalba** (α-Lactalbumin, wird zur Herstellung von Laktose gebraucht)? Sieh dir eins davon an:

In [ ]:
plotGene(dge, "Wap", group_by = "group")

**Frage:** Wap ist in säugenden Zellen signifikant höher, steht aber nicht oben in der Liste. Warum nicht?

✏️ **Deine Antwort:** *(Doppelklick hier, um zu schreiben)*



Jetzt die Gene, die **runter** gehen:

In [ ]:
showTop(fit, "lactating_vs_pregnant", n = 10, direction = "down")

### Dein Fazit

**Aufgabe:** Beantworte die Forschungsfrage in zwei oder drei Sätzen, so wie in einem Laborbericht: Was hast du verglichen, was hast du gefunden, und was bedeutet es?

✏️ **Deine Antwort:** *(Doppelklick hier, um zu schreiben)*



### Bonus: luminale gegen basale Zellen

Sieh dir den zweiten Kontrast an. Welche Gene machen eine basale Zelle zu einer basalen Zelle?

In [ ]:
showTop(fit, "___", n = 15)

✏️ **Deine Antwort:** *(Doppelklick hier, um zu schreiben)*



**Gut gemacht!** 🎉 Du hast eine komplette differenzielle Expressionsanalyse durchgeführt, genau so, wie Forschende es tun. Jetzt bist du dran: Wähle eines der Notebooks `02_diy_…`, `03_diy_…` oder `04_diy_…` und beantworte selbstständig eine neue Forschungsfrage.

## Quellen

**Daten**
* **Milch (Tutorial):** Fu NY, Rios AC, Pal B, et al. (2015). EGF-mediated induction of Mcl-1 at the switch to lactation is essential for alveolar cell survival. *Nature Cell Biology* 17:365–375. [doi:10.1038/ncb3117](https://doi.org/10.1038/ncb3117). Counts aus GEO [GSE60450](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE60450).
* **Dexamethason (airway):** Himes BE, Jiang X, Wagner P, et al. (2014). RNA-Seq transcriptome profiling identifies CRISPLD2 as a glucocorticoid responsive gene that modulates cytokine function in airway smooth muscle cells. *PLoS ONE* 9(6):e99625. [doi:10.1371/journal.pone.0099625](https://doi.org/10.1371/journal.pone.0099625). Counts aus dem Bioconductor-Paket [airway](https://bioconductor.org/packages/airway) (Love M).
* **Pasilla:** Brooks AN, Yang L, Duff MO, et al. (2011). Conservation of an RNA regulatory map between *Drosophila* and mammals. *Genome Research* 21:193–202. [doi:10.1101/gr.108662.110](https://doi.org/10.1101/gr.108662.110). Counts aus dem Bioconductor-Paket [pasilla](https://bioconductor.org/packages/pasilla).
* **Brustkrebs:** The Cancer Genome Atlas Network (2012). Comprehensive molecular portraits of human breast tumours. *Nature* 490:61–70. [doi:10.1038/nature11412](https://doi.org/10.1038/nature11412). Counts (STAR – Counts) von 10 Patientinnen aus TCGA-BRCA, aus den [NCI Genomic Data Commons](https://portal.gdc.cancer.gov).
* **Gennamen:** Bioconductor-Annotationspakete org.Mm.eg.db, org.Hs.eg.db, org.Dm.eg.db (Carlson M).

**Software und Methode**
* **limma:** Ritchie ME, Phipson B, Wu D, et al. (2015). limma powers differential expression analyses for RNA-sequencing and microarray studies. *Nucleic Acids Research* 43(7):e47. [doi:10.1093/nar/gkv007](https://doi.org/10.1093/nar/gkv007)
* **voom:** Law CW, Chen Y, Shi W, Smyth GK (2014). voom: precision weights unlock linear model analysis tools for RNA-seq read counts. *Genome Biology* 15:R29. [doi:10.1186/gb-2014-15-2-r29](https://doi.org/10.1186/gb-2014-15-2-r29)
* **edgeR:** Chen Y, Chen L, Lun ATL, Baldoni PL, Smyth GK (2025). edgeR v4: powerful differential analysis of sequencing data with expanded functionality and improved support for small counts and larger datasets. *Nucleic Acids Research* 53(2):gkaf018. [doi:10.1093/nar/gkaf018](https://doi.org/10.1093/nar/gkaf018)
* **Ablauf, dem dieses Praktikum folgt:** Law CW, Alhamdoosh M, Su S, et al. (2016). RNA-seq analysis is easy as 1-2-3 with limma, Glimma and edgeR. *F1000Research* 5:1408. [doi:10.12688/f1000research.9005.3](https://doi.org/10.12688/f1000research.9005.3)